# 📖 Notebook 5: Observability — Monitoring with Prometheus and Grafana

You can't fix what you can't see. In this notebook you'll install a production-grade monitoring stack
inside your minikube cluster and learn how to collect metrics, build dashboards, and create alerts.

We'll use the same tools that Netflix, Spotify, and most Fortune 500 companies rely on:
**Prometheus** for collecting metrics and **Grafana** for visualizing them.

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the three pillars of observability: metrics, logs, and traces
- Enable **Metrics Server** and use `kubectl top` to inspect resource usage
- Install **Prometheus + Grafana** via Helm (kube-prometheus-stack)
- Query Prometheus for built-in Kubernetes metrics
- Create a **ServiceMonitor** so Prometheus scrapes your own app's `/metrics` endpoint
- Access **Grafana** dashboards to visualize cluster and application health
- Write a **PrometheusRule** alert that fires when a service goes down

## 🛠️ Setup

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

Make sure minikube is running and your sample apps from Notebook 02 are deployed in the `k8s-lab` namespace.

In [ ]:
# Verify cluster is running and apps are deployed
!minikube status
!echo '---'
!kubectl get deployments -n k8s-lab

## 🔭 The Three Pillars of Observability

When something goes wrong in production, you need three types of data to find the problem:

| Pillar | What It Tells You | Example Tool |
|--------|-------------------|-------------|
| **Metrics** | Numbers over time — CPU, memory, request count, error rate | Prometheus |
| **Logs** | Detailed text records of what happened | Loki, ELK |
| **Traces** | The path a request takes across multiple services | Jaeger, Tempo |

In this lab we focus on **Metrics** — the most important pillar for keeping a cluster healthy.

```
Your App ──▶ /metrics endpoint ──▶ Prometheus (scrapes every 30s) ──▶ Grafana (dashboards)
                                         │
                                         ▼
                                   AlertManager ──▶ Slack / PagerDuty
```

## 📏 Step 1: Enable Metrics Server

**Metrics Server** is a lightweight component that collects CPU and memory usage from every node and pod.
It powers two things:
- `kubectl top` — see resource usage from the command line
- **HPA** (Horizontal Pod Autoscaler) — automatically scale pods based on CPU/memory

Minikube makes it easy — just enable the addon:

In [ ]:
# Enable metrics server
!minikube addons enable metrics-server

# Wait for it to be ready (takes ~30 seconds)
!kubectl wait --for=condition=ready pod -l k8s-app=metrics-server -n kube-system --timeout=90s

In [ ]:
# Now we can see resource usage!
# Node-level: how much CPU and memory is the whole node using?
!kubectl top nodes

print()

# Pod-level: which pods are using the most resources?
!kubectl top pods -n k8s-lab

**What you see**: CPU in millicores (1000m = 1 full CPU core) and Memory in Mi/Gi.

⚠️ **Important**: Metrics Server is for `kubectl top` and HPA only. It keeps only the *latest* data point — no history. For real monitoring, you need Prometheus.

## 📊 Step 2: Install Prometheus + Grafana

The **kube-prometheus-stack** Helm chart installs everything you need in one command:

```
┌──────────────────────────────────────────────────────────────┐
│  kube-prometheus-stack (Helm chart)                          │
│                                                              │
│  ┌─────────────┐  ┌──────────────┐  ┌───────────────────┐  │
│  │ Prometheus   │  │ Grafana       │  │ AlertManager      │  │
│  │ (collects    │  │ (visualizes   │  │ (sends alerts     │  │
│  │  metrics)    │  │  dashboards)  │  │  to Slack, etc.)  │  │
│  └──────┬──────┘  └──────────────┘  └───────────────────┘  │
│         │                                                    │
│  ┌──────▼──────┐  ┌──────────────┐                          │
│  │ node-exporter│  │kube-state-   │                          │
│  │ (OS metrics) │  │metrics       │                          │
│  │              │  │(K8s objects) │                          │
│  └─────────────┘  └──────────────┘                          │
└──────────────────────────────────────────────────────────────┘
```

- **Prometheus** scrapes `/metrics` endpoints from all pods every 30 seconds
- **node-exporter** exposes OS-level metrics (disk, network, CPU per core)
- **kube-state-metrics** exposes Kubernetes object states (how many pods are running, pending, failed)
- **Grafana** provides pre-built dashboards for all of the above
- **AlertManager** routes alerts to your notification channels

In [ ]:
# Add the Prometheus community Helm repo
!helm repo add prometheus-community https://prometheus-community.github.io/helm-charts
!helm repo update

In [ ]:
# Install the stack with our custom values (small resource footprint for minikube)
!helm install prometheus prometheus-community/kube-prometheus-stack \
    -n monitoring --create-namespace \
    -f ../manifests/prometheus-values.yaml \
    --wait --timeout 5m

print("\n✅ Prometheus stack installed!")

In [ ]:
# Check all the pods that were created
!kubectl get pods -n monitoring

print("\nYou should see: prometheus, grafana, alertmanager, node-exporter, kube-state-metrics")

## 🔍 Step 3: Query Prometheus

Prometheus has its own query language called **PromQL**. Let's access it.

We'll use `kubectl port-forward` to access the Prometheus web UI from our browser.

In [ ]:
# Start port-forward in background so we can access Prometheus
import subprocess, time

proc = subprocess.Popen(
    ["kubectl", "port-forward", "-n", "monitoring",
     "svc/prometheus-kube-prometheus-prometheus", "9090:9090"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print("✅ Prometheus UI available at: http://localhost:9090")
print("   Open it in your browser and try these PromQL queries:")
print()
print('   1. up                                    — which targets are being scraped?')
print('   2. kube_pod_status_phase{namespace="k8s-lab"}  — pod phases in our namespace')
print('   3. container_cpu_usage_seconds_total       — CPU usage by container')
print('   4. node_memory_MemAvailable_bytes          — available memory on the node')

### 💡 Common PromQL Queries

| Query | What It Shows |
|-------|---------------|
| `up` | All scrape targets and whether they're reachable |
| `rate(container_cpu_usage_seconds_total[5m])` | CPU usage rate over 5 minutes |
| `container_memory_working_set_bytes` | Current memory usage per container |
| `kube_deployment_status_replicas_available` | How many replicas are running per deployment |
| `sum(rate(http_requests_total[5m])) by (service)` | Request rate per service |

## 🎯 Step 4: Expose Your App's Metrics

Our sample apps already expose a `/metrics` endpoint (check `apps/user-service/app.py`).
But Prometheus doesn't know about it yet — we need to tell it to scrape our app.

In the kube-prometheus-stack, you do this by creating a **ServiceMonitor** — a custom resource
that tells Prometheus: "hey, scrape this Service's pods on this port and path."

```
ServiceMonitor ──▶ tells Prometheus ──▶ scrape user-service pods on port 8001 at /metrics
```

In [ ]:
%%writefile /tmp/servicemonitor.yaml
apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: user-service-monitor
  namespace: k8s-lab
  labels:
    release: prometheus
spec:
  selector:
    matchLabels:
      app: user-service
  endpoints:
    - port: "8001"
      path: /metrics
      interval: 15s

In [ ]:
!kubectl apply -f /tmp/servicemonitor.yaml

print("\n✅ ServiceMonitor created!")
print("Wait ~30 seconds, then check Prometheus targets:")
print("   http://localhost:9090/targets — look for 'serviceMonitor/k8s-lab/user-service-monitor'")

In [ ]:
# Verify our app metrics are being scraped
# First, let's see what our app exposes:
!kubectl exec -n k8s-lab deploy/user-service -- \
    wget -qO- http://localhost:8001/metrics 2>/dev/null || \
    echo "(If this fails, verify user-service is running: kubectl get pods -n k8s-lab)"

## 📈 Step 5: Access Grafana Dashboards

Grafana is the visualization layer. The kube-prometheus-stack comes with dozens of
pre-built dashboards for Kubernetes.

Let's access it:

In [ ]:
# Port-forward Grafana
import subprocess, time

grafana_proc = subprocess.Popen(
    ["kubectl", "port-forward", "-n", "monitoring",
     "svc/prometheus-grafana", "3000:80"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print("✅ Grafana available at: http://localhost:3000")
print()
print("   Login credentials:")
print("   Username: admin")
print("   Password: admin  (set in prometheus-values.yaml)")
print()
print("   Try these dashboards (search in the dashboard menu):")
print("   1. 'Kubernetes / Compute Resources / Namespace (Pods)'")
print("   2. 'Kubernetes / Compute Resources / Node (Pods)'")
print("   3. 'Node Exporter / Nodes'")

### 🏠 Built-in Dashboards Worth Exploring

| Dashboard | What It Shows |
|-----------|---------------|
| Kubernetes / Compute Resources / Namespace | CPU, memory, network per namespace |
| Kubernetes / Compute Resources / Pod | CPU, memory, network per pod |
| Kubernetes / Networking / Namespace | Bandwidth, packet drops |
| Node Exporter / Nodes | OS-level: disk, CPU cores, memory, network |
| CoreDNS | DNS query rate, latency, errors |

## 🚨 Step 6: Create an Alert Rule

What good is monitoring if nobody gets notified when something breaks?

A **PrometheusRule** defines conditions that should trigger an alert.
Let's create one: "alert me if any of our sample services is down for more than 2 minutes."

In [ ]:
%%writefile /tmp/alert-rule.yaml
apiVersion: monitoring.coreos.com/v1
kind: PrometheusRule
metadata:
  name: app-alerts
  namespace: monitoring
  labels:
    release: prometheus
spec:
  groups:
    - name: k8s-lab.rules
      rules:
        - alert: ServiceDown
          expr: up{namespace="k8s-lab"} == 0
          for: 2m
          labels:
            severity: critical
          annotations:
            summary: "Service {{ $labels.job }} is down"
            description: "{{ $labels.job }} in namespace {{ $labels.namespace }} has been down for more than 2 minutes."

In [ ]:
!kubectl apply -f /tmp/alert-rule.yaml

print("\n✅ Alert rule created!")
print("   Check it in Prometheus: http://localhost:9090/alerts")

### 🧪 Exercise: Trigger the Alert

Let's deliberately break something and see the alert fire:

1. Scale user-service to 0 replicas
2. Wait ~2 minutes
3. Check the alert status in Prometheus
4. Fix it by scaling back up

In [ ]:
# Break it! Scale to zero replicas
!kubectl scale deployment user-service -n k8s-lab --replicas=0

print("\n💥 user-service scaled to 0 — no pods running!")
print("   Wait 2 minutes, then check http://localhost:9090/alerts")
print("   You should see 'ServiceDown' in PENDING or FIRING state.")

In [ ]:
# Fix it! Scale back up
!kubectl scale deployment user-service -n k8s-lab --replicas=2
!kubectl wait --for=condition=ready pod -l app=user-service -n k8s-lab --timeout=60s

print("\n✅ user-service is back! Alert will auto-resolve.")

## 🧹 Clean Up

The monitoring stack uses significant resources. Remove it if you need to free up memory for later labs.

In [ ]:
# Stop port-forwards
try:
    proc.terminate()
    grafana_proc.terminate()
except:
    pass

# Optional: remove the monitoring stack to free resources
# !helm uninstall prometheus -n monitoring
# !kubectl delete namespace monitoring

# Clean up temp files
!rm -f /tmp/servicemonitor.yaml /tmp/alert-rule.yaml

print("✅ Cleaned up! (monitoring namespace kept for later labs — uncomment above to remove)")

## 🎓 What You Learned

In this notebook you:

1. **Enabled Metrics Server** and used `kubectl top` to see CPU and memory usage
2. **Installed Prometheus + Grafana** using the kube-prometheus-stack Helm chart
3. **Queried Prometheus** with PromQL for built-in Kubernetes metrics
4. **Created a ServiceMonitor** to scrape your application's custom `/metrics` endpoint
5. **Explored Grafana dashboards** for cluster-wide and namespace-level observability
6. **Created a PrometheusRule** alert that fires when a service goes down
7. **Triggered and resolved** an alert by scaling a deployment to zero and back

### Key Takeaways

- **Metrics Server** is for `kubectl top` and HPA — it has no history
- **Prometheus** stores time-series data and is the backbone of Kubernetes monitoring
- **ServiceMonitor** is how you tell Prometheus to scrape your app — no config file editing
- **Grafana** comes with dozens of pre-built dashboards for Kubernetes
- **PrometheusRule** defines alerting conditions — always create alerts for critical services

### Next Steps

In **Notebook 06** you'll learn how to secure your cluster with RBAC, NetworkPolicies,
and Pod Security Standards — so different teams can safely share the same cluster.